# Боты в трафике Авито

Score от 0 до 1 на каждую куку. Порог не отправляю, его подберет проверка. Свою цифру считаю `metric.py`, это их функция precision при recall от 70%.

Тест позже трейна: трейн 6-19 апреля, тест 20-26. Валидацию взяла тоже по дате, с 17 апреля, последние 3 дня трейна. Random split не стал делать, он тут смотрит в будущее.

В признаки кладу только события из окна `[window_start_ts, window_end_ts)`. Капчу смотрела отдельно: в файле она есть, но вся вне окна, внутри окна нули, так что толку от нее нет.

По трейну у ботов паузы короче, категорий меньше, городов больше. Один и тот же запрос часто в нескольких городах, страницу выдачи листают глубже. Фото и избранное относительно просмотров реже. Куки помоложе.

`user_agent` целиком не кодировала. Там куча строк и разный регистр. Оставил флаги headless, scrapy, curl, python и длину. Курсор на android/ios пустой у всех, и у людей тоже, поэтому долю курсора считаю только на web/desktop.

Модель lightgbm, деревья мелкие: `num_leaves=15`, `min_child_samples=80`, 150 деревьев, `learning_rate=0.05`, `random_state=42`. Ботов примерно 8%, глубокое дерево на этом быстро запоминает трейн. Лес из quickstart на числе событий и числе объявлений на тех же днях дает около 0.10. Бустинг около 0.82.

С 100 до 200 деревьев цифра почти не двигалась, early stopping не включала. Пробовала выбрать число деревьев на 15-16 апреля и смотреть 17-19, получалось около 0.80, то есть не от подгонки под один кусок.

На сабмит учу уже на всем трейне. `n_jobs=1` и `deterministic=True`, без этого lightgbm от запуска к запуску чуть меняет score.

Python 3.11.8. pandas 2.2.2, numpy 2.4.4, scikit-learn 1.8.0, lightgbm 4.7.0. Ноутбук класть рядом с `metric.py` и папкой `data`.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

from metric import precision_at_recall

SEED = 42
DATA_DIR = Path('data')

train = pd.read_csv(DATA_DIR / 'train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv(DATA_DIR / 'test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv(DATA_DIR / 'events.csv.gz', parse_dates=['event_ts'])

print(train.shape, test.shape, events.shape)
print('доля ботов:', round(float(train.target.mean()), 4))
print('трейн', train.window_start_ts.min().date(), '-', train.window_start_ts.max().date())
print('тест ', test.window_start_ts.min().date(), '-', test.window_start_ts.max().date())
print('общие cookie_id:', len(set(train.cookie_id) & set(test.cookie_id)))


## Окно

Беру `window_start_ts <= event_ts < window_end_ts`, как в условии.

Полные дубли строк выкинула. Их немного, у ботов и людей доля похожая, но из-за копии пауза между событиями становится 0.

Платформа в файле написана как web, Web, WEB. Привожу к lower. iphone склеиваю с ios.


In [ ]:
EVENT_NAMES = [
    'item_view',
    'search_results_view',
    'photo_swipe',
    'favorite_add',
    'seller_page_view',
    'contact_phone_show',
    'contact_chat_open',
    'contact_message_sent',
    'login',
]


def events_in_window(events, meta):
    cols = ['cookie_id', 'window_start_ts', 'window_end_ts']
    ev = events.merge(meta[cols], on='cookie_id', how='inner')
    mask = (ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)
    ev = ev.loc[mask].drop(columns=['window_start_ts', 'window_end_ts'])
    ev = ev.drop_duplicates()
    return ev


def _entropy(s):
    vc = s.dropna()
    if vc.empty:
        return np.nan
    p = vc.value_counts(normalize=True).to_numpy()
    return float(-(p * np.log(p)).sum())


def _top_share(s):
    # 1, если все события в одной категории или одном городе
    vc = s.dropna()
    if vc.empty:
        return np.nan
    return float(vc.value_counts().iloc[0] / len(vc))


def _ua_flags(ua):
    # ua как категорию не беру, разных строк слишком много
    u = ua.fillna('').str.lower()
    return pd.DataFrame({
        'ua_headless': u.str.contains('headless').astype(np.int8),
        'ua_scrapy': u.str.contains('scrapy').astype(np.int8),
        'ua_curl': u.str.contains('curl/').astype(np.int8),
        'ua_python': u.str.contains('python').astype(np.int8),
        'ua_okhttp': u.str.contains('okhttp').astype(np.int8),
        'ua_linux': u.str.contains('linux').astype(np.int8),
        'ua_len': ua.fillna('').str.len().astype(np.int16),
    }, index=ua.index)


ev = events_in_window(events, train)
print('событий в окне:', len(ev), 'из', len(events))
print(ev.event_name.value_counts())
print()
print(ev.platform.str.lower().value_counts())


## Признаки

Все считается по одной куке внутри ее окна.

Счетчики событий и доли: просмотр, поиск, фото, избранное, продавец, телефон, чат, сообщение, логин. Плюс сколько разных объявлений, категорий, городов, запросов.

По поиску смотрю макс и среднюю страницу выдачи, и сколько городов у одного и того же запроса. Это как раз случай, когда один текст гоняют по разным городам.

По времени: медиана паузы, разброс, доля пауз короче 15 секунд, событий в минуту, доля ночи с 0 до 6.

Если типа события не было, ставлю 0. Если паузу не из чего считать (одно событие) или поиска не было, оставляю NaN. В отношениях в знаменателе `+ 1`, иначе на нулевых просмотрах будет inf.

Возраст куки беру на начало окна. `cookie_created_at` к этому моменту уже известен.


In [ ]:
def build_features(events, meta):
    ev = events_in_window(events, meta)
    ev = ev.sort_values(['cookie_id', 'event_ts'], kind='mergesort')

    ev['plat'] = ev.platform.str.lower().replace({'iphone': 'ios'})
    ev['is_web'] = ev.plat.isin(['web', 'desktop']).astype(np.int8)
    ev['has_ptr'] = ev.pointer_x.notna().astype(np.int8)
    ev['hour'] = ev.event_ts.dt.hour.astype(np.int8)
    ev['is_night'] = ev.hour.between(0, 6).astype(np.int8)
    ev['dt'] = ev.groupby('cookie_id', sort=False).event_ts.diff().dt.total_seconds()

    # подряд поиск-поиск или просмотр-просмотр, без фото между ними
    prev_name = ev.groupby('cookie_id', sort=False).event_name.shift()
    ev['pair_ss'] = ((prev_name == 'search_results_view') & (ev.event_name == 'search_results_view')).astype(np.int8)
    ev['pair_ii'] = ((prev_name == 'item_view') & (ev.event_name == 'item_view')).astype(np.int8)
    ev['pair_ip'] = ((prev_name == 'item_view') & (ev.event_name == 'photo_swipe')).astype(np.int8)

    prev_loc = ev.groupby('cookie_id', sort=False).item_location.shift()
    ev['loc_switch'] = (ev.item_location.notna() & prev_loc.notna() & ev.item_location.ne(prev_loc)).astype(np.int8)
    prev_cat = ev.groupby('cookie_id', sort=False).item_category.shift()
    ev['cat_switch'] = (ev.item_category.notna() & prev_cat.notna() & ev.item_category.ne(prev_cat)).astype(np.int8)

    ev = pd.concat([ev, _ua_flags(ev.user_agent)], axis=1)

    g = ev.groupby('cookie_id', sort=False)
    feat = pd.DataFrame({'n_events': g.size()})
    feat['n_items'] = g.item_id.nunique()
    feat['n_cat'] = g.item_category.nunique()
    feat['n_loc'] = g.item_location.nunique()
    feat['n_query'] = g.search_query.nunique()
    feat['n_seller'] = g.seller_type.nunique()
    feat['n_ua'] = g.user_agent.nunique()
    feat['n_plat'] = g.plat.nunique()
    feat['span_s'] = (g.event_ts.max() - g.event_ts.min()).dt.total_seconds()
    feat['max_page'] = g.search_page.max()
    feat['mean_page'] = g.search_page.mean()
    feat['ptr_share'] = g.has_ptr.mean()
    feat['web_share'] = g.is_web.mean()
    feat['android_share'] = g.plat.apply(lambda s: (s == 'android').mean())
    feat['ios_share'] = g.plat.apply(lambda s: (s == 'ios').mean())
    feat['night_share'] = g.is_night.mean()
    feat['n_hours'] = g.hour.nunique()
    feat['hour_entropy'] = g.hour.apply(_entropy)
    feat['cat_top_share'] = g.item_category.apply(_top_share)
    feat['loc_top_share'] = g.item_location.apply(_top_share)
    feat['query_top_share'] = g.search_query.apply(_top_share)
    feat['cat_entropy'] = g.item_category.apply(_entropy)
    feat['loc_entropy'] = g.item_location.apply(_entropy)
    feat['med_dt'] = g.dt.median()
    feat['mean_dt'] = g.dt.mean()
    feat['std_dt'] = g.dt.std()
    feat['p10_dt'] = g.dt.quantile(0.1)
    feat['fast_share'] = g.dt.apply(lambda s: (s < 15).mean() if s.notna().any() else np.nan)
    feat['slow_share'] = g.dt.apply(lambda s: (s > 600).mean() if s.notna().any() else np.nan)
    feat['pair_ss'] = g.pair_ss.mean()
    feat['pair_ii'] = g.pair_ii.mean()
    feat['pair_ip'] = g.pair_ip.mean()
    feat['loc_switch'] = g.loc_switch.mean()
    feat['cat_switch'] = g.cat_switch.mean()
    feat['ua_headless'] = g.ua_headless.max()
    feat['ua_scrapy'] = g.ua_scrapy.max()
    feat['ua_curl'] = g.ua_curl.max()
    feat['ua_python'] = g.ua_python.max()
    feat['ua_okhttp'] = g.ua_okhttp.max()
    feat['ua_linux'] = g.ua_linux.max()
    feat['ua_len'] = g.ua_len.median()
    feat['pro_share'] = g.seller_type.apply(lambda s: (s == 'pro').mean())
    feat['q_len'] = g.search_query.apply(lambda s: s.dropna().str.len().mean())

    for name in EVENT_NAMES:
        feat[f'c_{name}'] = ev.loc[ev.event_name == name].groupby('cookie_id').size()

    # один и тот же запрос: сколько страниц и в скольких городах
    search = ev.loc[ev.event_name == 'search_results_view', ['cookie_id', 'search_query', 'search_page', 'item_location']]
    if len(search):
        per_q = search.groupby(['cookie_id', 'search_query'], dropna=False).agg(
            n_pages=('search_page', 'nunique'),
            n_locs=('item_location', 'nunique'),
        ).reset_index()
        by_cookie = per_q.groupby('cookie_id')
        feat['max_pages_per_query'] = by_cookie.n_pages.max()
        feat['mean_pages_per_query'] = by_cookie.n_pages.mean()
        feat['max_locs_per_query'] = by_cookie.n_locs.max()
        feat['mean_locs_per_query'] = by_cookie.n_locs.mean()
    else:
        feat['max_pages_per_query'] = np.nan
        feat['mean_pages_per_query'] = np.nan
        feat['max_locs_per_query'] = np.nan
        feat['mean_locs_per_query'] = np.nan

    web = ev.loc[ev.is_web == 1]
    if len(web):
        feat['web_ptr_share'] = web.groupby('cookie_id').has_ptr.mean()
        feat['web_ptr_std'] = web.groupby('cookie_id').pointer_x.std()
    else:
        feat['web_ptr_share'] = np.nan
        feat['web_ptr_std'] = np.nan

    feat = feat.reindex(meta.cookie_id.values)

    zero_cols = [
        'n_events', 'n_items', 'n_cat', 'n_loc', 'n_query', 'n_seller', 'n_ua', 'n_plat',
        'span_s', 'n_hours', 'ptr_share', 'web_share', 'android_share', 'ios_share',
        'night_share', 'pair_ss', 'pair_ii', 'pair_ip', 'loc_switch', 'cat_switch',
        'ua_headless', 'ua_scrapy', 'ua_curl', 'ua_python', 'ua_okhttp', 'ua_linux',
    ] + [f'c_{name}' for name in EVENT_NAMES]
    feat[zero_cols] = feat[zero_cols].fillna(0)

    views = feat['c_item_view']
    feat['photo_per_view'] = feat['c_photo_swipe'] / (views + 1)
    feat['fav_per_view'] = feat['c_favorite_add'] / (views + 1)
    contacts = feat['c_contact_phone_show'] + feat['c_contact_chat_open'] + feat['c_contact_message_sent']
    feat['contact_per_view'] = contacts / (views + 1)
    feat['seller_per_view'] = feat['c_seller_page_view'] / (views + 1)
    feat['search_per_view'] = feat['c_search_results_view'] / (views + 1)
    feat['loc_per_cat'] = feat['n_loc'] / (feat['n_cat'] + 1)
    feat['item_per_cat'] = feat['n_items'] / (feat['n_cat'] + 1)
    feat['item_per_loc'] = feat['n_items'] / (feat['n_loc'] + 1)
    feat['page_per_query'] = feat['max_page'] / (feat['n_query'] + 1)
    feat['rate_per_min'] = feat['n_events'] / (feat['span_s'] / 60.0 + 1)
    feat['cv_dt'] = feat['std_dt'] / feat['mean_dt']
    feat['events_per_item'] = feat['n_events'] / (feat['n_items'] + 1)
    # если веба не было, web_ptr пустой. fillna(1), чтобы не пометить это как бота без мыши
    feat['web_and_nopointer'] = ((feat['web_share'] > 0.5) & (feat['web_ptr_share'].fillna(1) < 0.2)).astype(np.int8)

    age_s = (meta.window_start_ts - meta.cookie_created_at).dt.total_seconds().to_numpy()
    feat['age_days'] = age_s / 86400.0
    feat['age_lt_1d'] = (feat['age_days'] < 1).astype(np.int8)
    feat['age_lt_7d'] = (feat['age_days'] < 7).astype(np.int8)

    for name in EVENT_NAMES:
        feat[f's_{name}'] = feat[f'c_{name}'] / (feat['n_events'] + 1)

    feat.index.name = 'cookie_id'
    return feat


x_train = build_features(events, train)
x_test = build_features(events, test)
y = train.target.to_numpy()
cols = list(x_train.columns)

peek = x_train[['n_events', 'n_cat', 'n_loc', 'max_page', 'med_dt', 'max_locs_per_query', 'age_days']].copy()
peek['target'] = y
print(peek.groupby('target').median().round(2))
print('признаков:', len(cols))


## Валидация

С 17 по 19 апреля. Это хвост трейна, по датам ближе к тесту. Baseline считаю на этих же днях.

`random_state=42`, `n_jobs=1`, `deterministic=True`.


In [ ]:
valid = train.window_start_ts.ge('2026-04-17').to_numpy()
print('кук в валидации:', int(valid.sum()), 'ботов:', int(y[valid].sum()))

base_cols = ['n_events', 'n_items']
forest = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=0, n_jobs=1)
forest.fit(x_train.iloc[~valid][base_cols].fillna(0), y[~valid])
base_score = forest.predict_proba(x_train.iloc[valid][base_cols].fillna(0))[:, 1]

model = LGBMClassifier(
    n_estimators=150,
    learning_rate=0.05,
    num_leaves=15,
    min_child_samples=80,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    reg_alpha=0.1,
    objective='binary',
    random_state=SEED,
    n_jobs=1,
    verbose=-1,
    deterministic=True,
    force_col_wise=True,
    bagging_seed=SEED,
    feature_fraction_seed=SEED,
)
model.fit(x_train.iloc[~valid][cols], y[~valid])
valid_score = model.predict_proba(x_train.iloc[valid][cols])[:, 1]


def report(name, score):
    print(
        f'{name}: P@R70={precision_at_recall(y[valid], score):.4f}  '
        f'PR-AUC={average_precision_score(y[valid], score):.4f}  '
        f'ROC-AUC={roc_auc_score(y[valid], score):.4f}'
    )


report('baseline, случайный лес', base_score)
report('lightgbm', valid_score)


## Сабмит

Ту же модель учу на всем трейне. В файле `cookie_id` и `score`, порядок как в `test.csv`.


In [ ]:
model.fit(x_train[cols], y)
submission = pd.DataFrame({
    'cookie_id': test.cookie_id.to_numpy(),
    'score': model.predict_proba(x_test[cols])[:, 1],
})

assert len(submission) == len(test)
assert submission.cookie_id.is_unique
assert set(submission.cookie_id) == set(test.cookie_id)
assert submission.score.between(0, 1).all() and submission.score.notna().all()

submission.to_csv('submission.csv', index=False)
print(submission.head().to_string(index=False))
print('строк', len(submission))


## Где модель промахивается

Плохо ловит ботов, которые ходят медленно, в паре категорий и не листают выдачу далеко. По валидации ошибки в основном такие. Headless и scrapy в ua почти точно бот, но таких кук мало.

Если бот начнет двигать мышь и сидеть в одном городе, эти признаки развалятся. События вне окна не брал.
